In [16]:
import os
import warnings
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

In [17]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [18]:
DATA_PATH       = "/kaggle/input/datasets/quanghuylt/combined/combined.csv"   
METRICS_DIR     = "/kaggle/working/reports/metrics/arimax/"  # save metrics of a single ticker's predictions when trained by OOF
RESULTS_DIR     = "/kaggle/working/data/meta_test/"  # save final arimax predicted results on the test set, used to test the meta-learner
FIGURES_DIR     = "/kaggle/working/reports/figures/"   
MODEL_DIR       = "/kaggle/working/models/arimax/"   
OOF_DIR         = "/kaggle/working/data/meta_train/"  # save arimax predicted results to train meta-learner
SUMMARY_METRICS = "/kaggle/working/reports/metrics/arimax/ARIMAX_all_tickers.csv"  # save metrics of all tickers' predictions on test set after refitted on full training set

SYMBOLS   = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
EXOG_COLS = ["dxy", "fed_rate", "gold", "oil", "sp500"]  # macro exogenous variables

# fix 5
TRAIN_RATIO = 0.8
N_FOLDS     = 10 
GAP_DAYS    = 10   # purge gap between train end and val start to avoid leakage
# more folds (less data is used in the first train oof) 
# less gap days to avoid nan input data for meta lear

In [19]:
def split_data(df: pd.DataFrame):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    split_idx = int(len(df) * TRAIN_RATIO)
    train_df  = df.iloc[:split_idx].reset_index(drop=True)
    test_df   = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df

In [20]:
# fix 2
def scale_data(train_df, test_df):
    close_scaler = MinMaxScaler()
    exog_scaler  = MinMaxScaler()

    train_df["Close"]       = close_scaler.fit_transform(train_df[["Close"]])
    train_df[EXOG_COLS]     = exog_scaler.fit_transform(train_df[EXOG_COLS])

    test_df["Close"]        = close_scaler.transform(test_df[["Close"]])
    test_df[EXOG_COLS]      = exog_scaler.transform(test_df[EXOG_COLS])

    return train_df, test_df, close_scaler, exog_scaler

In [21]:
# fix 4
def prepare_exog(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[EXOG_COLS] = df[EXOG_COLS].diff()
    df = df.dropna().reset_index(drop=True)
    return df

In [22]:
def find_best_order(endog: pd.Series, exog: pd.DataFrame) -> tuple:
    log.info("    Running auto_arima to find best (p,d,q)...")
    model = auto_arima(
        endog,
        exogenous=exog,
        seasonal=False,          
        test="adf", 
        d=None,
        stepwise=True,
        information_criterion="aic",
        error_action="ignore",
        suppress_warnings=True,
        trace=False,
        max_p=3, max_q=3,        # cap order to avoid overfitting
    )
    order = model.order
    log.info(f"Best order={order}")
    return order

In [23]:
def run_oof(train_df: pd.DataFrame, order: tuple, ticker: str) -> pd.DataFrame:
    n         = len(train_df)
    fold_size = n // (N_FOLDS + 1)
    oof_preds = np.full(n, np.nan)  # nan everywhere, only val windows will be filled

    log.info(f"    OOF: {N_FOLDS} folds, fold_size={fold_size}, gap={GAP_DAYS}")

    for k in range(1, N_FOLDS + 1):
        train_end = k * fold_size
        val_start = train_end + GAP_DAYS   # skip gap days to prevent leakage through rolling features
        val_end = (k + 1) * fold_size if k < N_FOLDS else n # since folds have equal size

        if val_end > n:
            break

        fold_train = train_df.iloc[:train_end]
        fold_val   = train_df.iloc[val_start:val_end]

        try:
            result = ARIMA(
                fold_train["Close"],
                exog=fold_train[EXOG_COLS],   # macro variables as exogenous regressors
                order=order,
            ).fit()


            # fix 3f
            result_applied = result.apply(
                fold_val["Close"],
                exog=fold_val[EXOG_COLS],
                refit=False,
            )
            
            oof_preds[val_start:val_end] = result_applied.fittedvalues.values

            mae = np.mean(np.abs(result_applied.fittedvalues.values - fold_val["Close"].values))
            log.info(f"      Fold {k}: train=[0:{train_end}]  "
                     f"val=[{val_start}:{val_end}]  MAE={mae:.4f}")

        except Exception as e:
            log.warning(f"      Fold {k} failed: {e}")
            oof_preds[val_start:val_end] = np.nan

    # calculate nan count due to oof training
    nan_count = np.isnan(oof_preds).sum()
    print(f'  oof nan: {nan_count}/{n} ({nan_count/n:.1%}) — only the first ~{fold_size + GAP_DAYS} rows are uncovered')

    return pd.DataFrame({
        "Ticker":                 ticker,
        "Date":                   train_df["Date"].values,
        "Close":                  train_df["Close"].values,
        "ARIMAX_Predicted_Close": oof_preds,   # nan at gap regions between folds
    })

In [24]:
def predict_test(final_result, test_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    # fix 1
    # use the model to predict the final test set without refitting the parameters
    result_applied = final_result.apply(
        test_df["Close"],
        exog=test_df[EXOG_COLS],
        refit=False
    )

    return pd.DataFrame({
        "Ticker":                 ticker,
        "Date":                   test_df["Date"].values,
        "Close":                  test_df["Close"].values,
        "ARIMAX_Predicted_Close": result_applied.fittedvalues.values,
    })

In [25]:
def calc_metrics(
    df: pd.DataFrame,
    actual_col: str = "Close",
    pred_col:   str = "ARIMAX_Predicted_Close",
    ticker_col: str = "Ticker",
) -> pd.DataFrame:

    # recurse per ticker when df contains multiple tickers
    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return (
            df.groupby(ticker_col)
              .apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None))
              .reset_index()
        )

    df = df.copy()
    df = df.dropna(subset=[actual_col, pred_col])   # drop nan rows from oof gap regions
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true + 1e-8))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)  # +1 up, -1 down, 0 flat
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)                     # directional accuracy

    mask = actual_dir != 0                                            # exclude flat days from tpa
    tpa  = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    # volatility rmse: compare de-meaned series so level bias doesn't inflate the error
    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE":    mse,
        "MAE":    mae,
        "MAPE":   mape,
        "RMSE":   rmse,
        "R2":     r2,
        "DA":     da,
        "TPA":    tpa,
        "V-RMSE": v_rmse,
    }])

In [26]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker  = df["Ticker"].iloc[0]
    df_plot = df.dropna(subset=["ARIMAX_Predicted_Close"])

    os.makedirs(save_dir, exist_ok=True)
    sns.set(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # top panel: actual vs predicted close price
    axes[0].plot(df_plot["Date"], df_plot["Close"],
                 label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(df_plot["Date"], df_plot["ARIMAX_Predicted_Close"],
                 label="ARIMAX Predicted", color="#d62728", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"ARIMAX predicted results: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis="x", rotation=30)

    # bottom panel: residual error per day
    error  = df_plot["Close"] - df_plot["ARIMAX_Predicted_Close"]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")   # green = under-predicted, red = over-predicted
    axes[1].bar(df_plot["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    log.info(f"    Plot saved: {save_path}")

In [27]:
def train_one_ticker(ticker: str) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    # fix 6
    # df = prepare_exog(df)

    train_df, test_df = split_data(df)

    train_df, test_df, close_scaler, exog_scaler = scale_data(train_df, test_df)
    
    train_df = prepare_exog(train_df)
    test_df = prepare_exog(test_df)

    log.info(f"  Train: {len(train_df)} rows  Test: {len(test_df)} rows")

    order = find_best_order(train_df["Close"], train_df[EXOG_COLS])
    oof_df = run_oof(train_df, order, ticker)

    # refit on full train set to use all available signal before predicting test
    log.info("    Refitting on full train set...")
    final_result = ARIMA(
        train_df["Close"],
        exog=train_df[EXOG_COLS],
        order=order,
    ).fit()
    

    train_refit_df = pd.DataFrame({
        "Ticker": ticker,
        "Date": train_df["Date"].values,
        "Close": train_df["Close"].values,
        "ARIMAX_Predicted_Close": final_result.fittedvalues.values,
    })
    
    train_refit_df[["ARIMAX_Predicted_Close"]] = close_scaler.inverse_transform(train_refit_df[["ARIMAX_Predicted_Close"]])
    train_refit_df[["Close"]] = close_scaler.inverse_transform(train_refit_df[["Close"]])
    
    refit_metrics_df = calc_metrics(train_refit_df)
    refit_metrics_df.insert(0, "Ticker", ticker)
    
    log.info(f"  Refit Train — MAE={refit_metrics_df['MAE'].iloc[0]:.2f} "
             f"DA={refit_metrics_df['DA'].iloc[0]:.2%}  "
             f"R2={refit_metrics_df['R2'].iloc[0]:.4f}")

    test_result_df = predict_test(final_result, test_df, ticker)

    mask_oof = oof_df["ARIMAX_Predicted_Close"].notna()
    oof_df.loc[mask_oof, ["ARIMAX_Predicted_Close"]] = close_scaler.inverse_transform(oof_df.loc[mask_oof, ["ARIMAX_Predicted_Close"]])
    oof_df[["Close"]] = close_scaler.inverse_transform(oof_df[["Close"]])

    # inverse minmax scaler to get the original values
    test_result_df[["ARIMAX_Predicted_Close"]] = close_scaler.inverse_transform(test_result_df[["ARIMAX_Predicted_Close"]])
    test_result_df[["Close"]] = close_scaler.inverse_transform(test_result_df[["Close"]])

    metrics_df = calc_metrics(oof_df)
    metrics_df.insert(0, "Ticker", ticker)

    log.info(f"  OOF  — MAE={metrics_df['MAE'].iloc[0]:.2f} "
             f"DA={metrics_df['DA'].iloc[0]:.2%}  "
             f"R2={metrics_df['R2'].iloc[0]:.4f}")

    test_metrics = calc_metrics(test_result_df)
    test_metrics.insert(0, "Ticker", ticker)   
    log.info(f"  Test — MAE={test_metrics['MAE'].iloc[0]:.2f} "
             f"DA={test_metrics['DA'].iloc[0]:.2%}  "
             f"R2={test_metrics['R2'].iloc[0]:.4f}")

    os.makedirs(METRICS_DIR, exist_ok=True)
    metrics_df.to_csv(
        os.path.join(METRICS_DIR, f"ARIMAX_{ticker}_metrics.csv"), index=False
    )

    os.makedirs(OOF_DIR, exist_ok=True)
    oof_df.to_csv(
        os.path.join(OOF_DIR, f"oof_arimax_{ticker}.csv"), index=False
    )

    os.makedirs(RESULTS_DIR, exist_ok=True)
    test_result_df.to_csv(
        os.path.join(RESULTS_DIR, f"ARIMAX_{ticker}_predictions.csv"), index=False
    )

    plot_results(test_result_df, save_dir=FIGURES_DIR)

    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"arimax_{ticker}.pkl"), "wb") as f:
        pickle.dump({
            "result":       final_result,
            "order":        order,
            "ticker":       ticker,
            "close_scaler": close_scaler,
            "exog_scaler":  exog_scaler,
        }, f)

    log.info(f"  All outputs saved for {ticker}")
    
    return test_metrics # get the metrics of predictions on the test set

In [28]:
all_metrics = []

for ticker in SYMBOLS:
    try:
        m = train_one_ticker(ticker)
        all_metrics.append(m)
    except Exception as e:
        log.error(f"{ticker} FAILED: {e}")

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
    summary_df.to_csv(SUMMARY_METRICS, index=False)
    log.info(f"\nSummary saved: {SUMMARY_METRICS}")
    print("\n" + summary_df.to_string(index=False))

log.info("Done.")

03:16:49  INFO      ========== VNM ==========
03:16:49  INFO        Train: 3258 rows  Test: 814 rows
03:16:49  INFO          Running auto_arima to find best (p,d,q)...
03:16:53  INFO      Best order=(0, 1, 0)
03:16:53  INFO          OOF: 10 folds, fold_size=296, gap=10
03:16:54  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0014
03:16:54  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0030
03:16:55  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0038
03:16:55  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0048
03:16:56  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0075
03:16:57  INFO            Fold 6: train=[0:1776]  val=[1786:2072]  MAE=0.0103
03:16:58  INFO            Fold 7: train=[0:2072]  val=[2082:2368]  MAE=0.0116
03:16:59  INFO            Fold 8: train=[0:2368]  val=[2378:2664]  MAE=0.0117
03:17:00  INFO            Fold 9: train=[0:2664]  val=[2674:2960]  MAE=0.0103
03:17:01  INFO            Fold 10: t

  oof nan: 396/3258 (12.2%) — only the first ~306 rows are uncovered


03:17:02  INFO        Refit Train — MAE=0.51 DA=41.23%  R2=0.9990
03:17:02  INFO        OOF  — MAE=0.71 DA=41.91%  R2=0.9842
03:17:02  INFO        Test — MAE=0.61 DA=43.17%  R2=0.9477
03:17:04  INFO          Plot saved: /kaggle/working/reports/figures/VNM.png
03:17:04  INFO        All outputs saved for VNM
03:17:04  INFO      ========== FPT ==========
03:17:04  INFO        Train: 3258 rows  Test: 814 rows
03:17:04  INFO          Running auto_arima to find best (p,d,q)...
03:17:11  INFO      Best order=(0, 1, 3)
03:17:11  INFO          OOF: 10 folds, fold_size=296, gap=10
03:17:11  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0011
03:17:12  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0009
03:17:12  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0017
03:17:14  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0016
03:17:15  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0016
03:17:17  INFO            Fold 6: train=

  oof nan: 396/3258 (12.2%) — only the first ~306 rows are uncovered


03:17:29  INFO        Refit Train — MAE=0.19 DA=45.84%  R2=0.9993
03:17:29  INFO        OOF  — MAE=0.24 DA=46.31%  R2=0.9945
03:17:29  INFO        Test — MAE=1.09 DA=49.45%  R2=0.9953
03:17:31  INFO          Plot saved: /kaggle/working/reports/figures/FPT.png
03:17:31  INFO        All outputs saved for FPT
03:17:31  INFO      ========== MSN ==========
03:17:31  INFO        Train: 3258 rows  Test: 814 rows
03:17:31  INFO          Running auto_arima to find best (p,d,q)...
03:17:57  INFO      Best order=(2, 1, 2)
03:17:57  INFO          OOF: 10 folds, fold_size=296, gap=10
03:17:57  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0106
03:17:59  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0073
03:17:59  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0053
03:18:02  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0048
03:18:03  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0039
03:18:06  INFO            Fold 6: train=

...

  oof nan: 396/3258 (12.2%) — only the first ~306 rows are uncovered


03:19:47  INFO        Refit Train — MAE=0.29 DA=42.98%  R2=0.9991
03:19:47  INFO        OOF  — MAE=0.38 DA=41.70%  R2=0.9892
03:19:47  INFO        Test — MAE=1.14 DA=49.20%  R2=0.9972
03:19:49  INFO          Plot saved: /kaggle/working/reports/figures/VIC.png
03:19:49  INFO        All outputs saved for VIC
03:19:49  INFO      ========== HPG ==========
03:19:49  INFO        Train: 3258 rows  Test: 814 rows
03:19:49  INFO          Running auto_arima to find best (p,d,q)...
03:20:12  INFO      Best order=(2, 1, 2)
03:20:12  INFO          OOF: 10 folds, fold_size=296, gap=10
03:20:12  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0007
03:20:13  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0007
03:20:14  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0013
03:20:15  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0017
03:20:16  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0020
03:20:18  INFO            Fold 6: train=

  oof nan: 396/3258 (12.2%) — only the first ~306 rows are uncovered


03:20:28  INFO        Refit Train — MAE=0.12 DA=44.83%  R2=0.9990
03:20:28  INFO        OOF  — MAE=0.15 DA=45.19%  R2=0.9931
03:20:28  INFO        Test — MAE=0.28 DA=46.00%  R2=0.9873
03:20:30  INFO          Plot saved: /kaggle/working/reports/figures/HPG.png
03:20:30  INFO        All outputs saved for HPG
03:20:30  INFO      
Summary saved: /kaggle/working/reports/metrics/arimax/ARIMAX_all_tickers.csv
03:20:30  INFO      Done.



Ticker      MSE      MAE     MAPE     RMSE       R2       DA      TPA   V-RMSE
   VNM 0.796889 0.613478 0.010104 0.892686 0.947679 0.431734 0.464901 0.892677
   FPT 2.515208 1.093741 0.012002 1.585941 0.995349 0.494465 0.516710 1.585676
   MSN 2.060457 1.033580 0.014008 1.435429 0.958078 0.483395 0.520530 1.435259
   VCB 0.777043 0.577198 0.009674 0.881500 0.953046 0.466175 0.508043 0.881440
   VIC 6.121385 1.138346 0.016445 2.474143 0.997247 0.492005 0.552486 2.463022
   HPG 0.156763 0.279815 0.012513 0.395933 0.987348 0.460025 0.496021 0.395705
